# Splunk SOAR: list all containers

Pages through `GET /rest/container` for one SOAR instance, oldest to newest, and saves the SOAR host and container ID of every container to a CSV.

The `ph-auth-token` is read from `SOAR1_TOKEN` in the repo-root `.env` file (git-ignored; see `../.env.example`).

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

SERVER = "https://prod-soar.mdr.tekstream.com"
load_dotenv(find_dotenv(usecwd=True))
TOKEN = os.environ.get("SOAR1_TOKEN", "")  # from .env

LOOKBACK_MONTHS = None  # None = every container; e.g. 5 = only containers created in the last 5 months
PAGE_SIZE = 1000

OUTPUT_DIR = Path("../data")

assert TOKEN, "set SOAR1_TOKEN in .env (see ../.env.example)"
SOAR_HOST = urlparse(SERVER).hostname

In [ ]:
session = requests.Session()
session.headers.update({"ph-auth-token": TOKEN, "Content-Type": "application/json"})

params = {"sort": "id", "order": "asc", "page_size": PAGE_SIZE}
if LOOKBACK_MONTHS:
    since = (pd.Timestamp.now(tz="UTC") - pd.DateOffset(months=LOOKBACK_MONTHS)).strftime("%Y-%m-%dT%H:%M:%SZ")
    params["_filter_create_time__gte"] = f'"{since}"'
    print(f"Containers created since {since}")

rows, page = [], 0
while True:
    r = session.get(f"{SERVER}/rest/container", params={**params, "page": page}, allow_redirects=False, timeout=300)
    r.raise_for_status()
    body = r.json()
    rows.extend({"container_id": c["id"], "create_time": c.get("create_time")} for c in body["data"])
    print(f"\rpage {page + 1}/{body['num_pages']}  {len(rows)}/{body['count']} containers", end="")
    page += 1
    if page >= body["num_pages"]:
        break
print()

In [ ]:
df = pd.DataFrame(rows).drop_duplicates("container_id")
df.insert(0, "soar_host", SOAR_HOST)
print(f"{len(df)} containers, created {df['create_time'].min()} to {df['create_time'].max()}")

OUTPUT_DIR.mkdir(exist_ok=True)
out_path = OUTPUT_DIR / f"containers_{SOAR_HOST}.csv"
df[["soar_host", "container_id"]].to_csv(out_path, index=False)
print(f"Wrote {out_path}")
df.head()